# 06 -- Lazy and Temporal Algebra

This notebook introduces `RasterExpression` (deferred calculation),
`ma.source()`, terrain expression nodes, `ma.explain()` / `ma.plan()`,
bounded-window writes with `ma.write()`, and temporal map algebra with
`ma.temporal_source()` and temporal reductions.  It draws from command-line
examples 27 and 31.

A `RasterExpression` describes a calculation without running it.  Use
`ma.explain()` and `ma.plan()` to review it, `ma.compute()` to
materialise it eagerly, or `ma.write()` for supported bounded execution.

**Run the individual scripts:** `27_map_algebra_terrain_resample.py`,
`31_map_algebra_temporal.py`

## Setup

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    """Find the Lunarscout repository root from the kernel working directory."""
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError(
        "Cannot locate Lunarscout repository root. "
        "Launch Jupyter from the repository root directory."
    )

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))
sys.path.insert(0, str(_REPO / "examples"))


import lunarscout as ls
import numpy as np

ma = ls.map_algebra

from _example_support import (
    ensure_synthetic_scenario, ensure_synthetic_series,
    synthetic_georef, synthetic_dem,
)

WORKSPACE = Path("/tmp/lunarscout_notebook_06")
WORKSPACE.mkdir(parents=True, exist_ok=True)

ensure_synthetic_scenario(WORKSPACE)
georef = synthetic_georef()
SCENARIO = ls.open_scenario(WORKSPACE / "synthetic_scenario")
series = ensure_synthetic_series(WORKSPACE)

DEM_PATH = SCENARIO.dem_path()
print(f"DEM:    {DEM_PATH}")
print(f"Grid:   {georef.width}x{georef.height}, {georef.pixel_size_x}m")
print(f"Series: {series.root}")

---
## 1. Expressions, Explain, and Plan

A `RasterExpression` is an immutable description of a calculation that has
not yet run.  You obtain one from `ma.source()`, `Raster.expression()`, or
registered operators.

In [ ]:
# Eager reference: slope from the DEM.
dem_arr, _ = ls.read_geotiff(DEM_PATH)
eager_slope, _ = ls.slope(dem_arr, georef, output_nodata=-9999.0)

# Lazy: describe the same calculation without running it.
dem_expr = ma.source(DEM_PATH, units="m")
slope_expr = ma.slope(dem_expr)

print(type(slope_expr).__name__)

In [ ]:
# Explain: a human-readable summary of the expression tree.
print(ma.explain(slope_expr))

**Explain** shows the operation tree without calculating any pixels.  Put
`ma.explain()` directly beside the expression it describes.

In [ ]:
# Plan: validate the expression and inspect the output strategy.
import tempfile
with tempfile.TemporaryDirectory() as td:
    out = Path(td) / "slope.tif"
    plan = ma.plan(slope_expr, output=out)
    print(f"Output grid:  {plan['output_grid']['width']}x{plan['output_grid']['height']}")
    print(f"Output dtype: {plan['output_dtype']}")
    print(f"Window size:  {plan['planner']['window_width']}x{plan['planner']['window_height']}")
    print(f"Window count: {plan['planner']['total_windows']}")
    print(f"Node count:   {plan['node_count']}")

Planning is read-only.  It rejects unsupported operations before creating
any output file.

**Try this:** explain `ma.hillshade(dem_expr, azimuth=315.0, altitude=45.0)`.
What nodes do you see?

---
## 2. Bounded-Window Writes

`ma.write()` evaluates an expression in bounded windows and produces a
staged, resumable, atomically-published GeoTIFF.

In [ ]:
_out_dir = WORKSPACE / "terrain_resample"
_out_dir.mkdir(parents=True, exist_ok=True)

# Compute and write slope through the expression path.
slope_out = _out_dir / "slope.tif"
ma.write(
    str(slope_out),
    slope_expr,
    dtype=np.float32,
    invalid_value=-9999.0,
    window_width=128,
    window_height=128,
    overwrite=True,
)
print(f"Wrote {slope_out}")

# Read back and compare with eager calculation.
read_slope, _ = ls.read_geotiff(slope_out)
# Strip nodata for valid-cell comparison.
eager_valid = eager_slope != -9999.0
same = np.allclose(eager_slope[eager_valid], read_slope[eager_valid], atol=1e-4)
print(f"Windowed result matches eager: {same}")

In [ ]:
# Compose terrain and illumination: weighted score.
illum_mean, _ = ls.temporal_mean(series)
sun_raster = ma.from_existing(illum_mean, georef, units="fraction", name="illumination_mean")

hillshade_expr = ma.hillshade(dem_expr, azimuth=315.0, altitude=45.0)

# Normalise both to [0, 1].
hs_norm = ma.normalize_minmax(hillshade_expr, minimum=0.0, maximum=1.0)
sun_norm = ma.normalize_minmax(
    sun_raster.expression(), minimum=0.0, maximum=1.0,
)

W_SUN, W_TERRAIN = 0.4, 0.6
score_expr = (W_SUN * sun_norm) + (W_TERRAIN * hs_norm)

print(ma.explain(score_expr))

In [ ]:
score_out = _out_dir / "combined_score.tif"
ma.write(
    str(score_out),
    score_expr,
    dtype=np.float32,
    invalid_value=-1.0,
    overwrite=True,
)
print(f"Wrote {score_out}")

The `invalid_value` (`-1.0`) identifies invalid payload bytes for
interchange.  The **dataset mask** remains the authoritative validity
representation.

**Try this:** inspect the slope plan again.  Then write hillshade in the
same way and verify the result against eager hillshade from `ls.hillshade()`.

---
## 3. Temporal Map Algebra

Wrap a file-backed `TemporalGeoTiffSeries` with `ma.temporal_source()`,
reduce it with `ma.temporal_mean()`, and combine the resulting spatial
expression with eager constraints.

In [ ]:
temporal_expr = ma.temporal_source(series)
mean_expr = ma.temporal_mean(temporal_expr)

print(ma.explain(mean_expr))

In [ ]:
# Temporal reductions are whole-raster nodes.  Use ma.compute() first.
mean_sun = ma.compute(mean_expr)

print(f"Mean sun: dtype={mean_sun.values.dtype}, "
      f"range=[{mean_sun.values.min():.4f}, {mean_sun.values.max():.4f}]")

In [ ]:
# Combine with a spatial constraint.
dem_arr, _ = ls.read_geotiff(DEM_PATH)
slope_vals, _ = ls.slope(dem_arr, georef, output_nodata=-9999.0)
slope_raster = ma.raster(slope_vals, georef, units="deg", name="slope")

SLOPE_MAX = 8.0        # degrees (illustrative)
SUN_MIN = 0.40         # fraction (illustrative)

candidate = (slope_raster <= SLOPE_MAX) & (mean_sun >= SUN_MIN)
print(f"Candidate cells: {candidate.values.sum()} / {candidate.values.size}")

In [ ]:
_temporal_out = SCENARIO.output_path("analysis/temporal")
_temporal_out.mkdir(parents=True, exist_ok=True)

ma.write(
    str(_temporal_out / "mean_sun.tif"),
    mean_sun.expression(),
    dtype=np.float32,
    invalid_value=-9999.0,
    overwrite=True,
)
ma.write(
    str(_temporal_out / "temporal_candidate.tif"),
    candidate.expression(),
    dtype=np.uint8,
    invalid_value=0,
    overwrite=True,
)
print(f"Temporal outputs written under {_temporal_out}")

The materialisation boundary is visible: the temporal source is
file-backed, the reduction streams layers, `ma.compute()` creates the
spatial result, and ordinary spatial map algebra handles the candidate mask.

Use a `TemporalRasterExpression` for temporal composition and a
`RasterExpression` after a time-reducing operation produces one spatial
field.

**Try this:** replace `temporal_mean` with `temporal_min` or
`temporal_max`.  How do the candidate counts change?

---
## Cleanup

In [ ]:
series.close()
print("Series closed.")